In [7]:
%load_ext autoreload
%autoreload 2


import numpy as np
import pandas as pd

from assignment_1.modules.util import (
    drop_zero_ratings,
    load_csv,
    random_split,
)

DATA_DIR = "../data"  # notebook runs from notebooks/

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [8]:
TASK_TRAIN_FILE = "tasks_1_2_train.csv"
TASK_TEST_FILE = "task_1_test.csv"
train = load_csv(TASK_TRAIN_FILE, DATA_DIR)
test = load_csv(TASK_TEST_FILE, DATA_DIR)

In [9]:
tr, val = random_split(train, val_frac=0.25, seed=0)
tr = drop_zero_ratings(tr)
val = drop_zero_ratings(val)
 
SCALE = (train.rating.min(), train.rating.max())

In [10]:
key = lambda d: d["user"] + "|" + d["item"]
print(f"test pairs seen in train: {key(test).isin(set(key(train))).mean():.2%}")

test pairs seen in train: 0.00%


In [11]:
print(test.columns.tolist())

z = train["rating"] == 0
print(f"zero rows: {z.mean():.2%} | users with any zero: {train.loc[z, 'user'].nunique() / train['user'].nunique():.1%}")

['user', 'item', 'rating']
zero rows: 1.48% | users with any zero: 23.7%


In [12]:
g = train[train["rating"] > 0].groupby("user")["rating"].agg(["count", "std", "nunique"])
w = test["user"].map(g["nunique"])
print(f"test rows from users with one distinct rating: {(w == 1).mean():.1%}")

test rows from users with one distinct rating: 1.3%


In [13]:
v = val.sort_values("timestamp")
nb = (tr.sort_values("timestamp")[["user", "timestamp", "rating"]]
        .rename(columns={"timestamp": "nb_ts", "rating": "nb_rating"}))
m = pd.merge_asof(v, nb, left_on="timestamp", right_on="nb_ts", by="user", direction="nearest")
m["gap"] = (m["timestamp"] - m["nb_ts"]).abs()

bins = [-1, 0, 60, 3600, 86400, 7 * 86400, 30 * 86400, np.inf]
labels = ["same second", "<1 min", "<1 h", "<1 day", "<1 wk", "<1 mo", "older"]
m["gap_bin"] = pd.cut(m["gap"], bins, labels=labels)
print(m.groupby("gap_bin", observed=True).apply(lambda d: pd.Series({
    "share": len(d) / len(m),
    "P(same rating)": (d["rating"] == d["nb_rating"]).mean(),
})).round(3))

             share  P(same rating)
gap_bin                           
same second  0.000           0.550
<1 min       0.006           0.546
<1 h         0.115           0.518
<1 day       0.158           0.494
<1 wk        0.420           0.472
<1 mo        0.192           0.448
older        0.109           0.426


In [14]:
test["rating"].describe(), test["rating"].value_counts().head()

(count    0.0
 mean     NaN
 std      NaN
 min      NaN
 25%      NaN
 50%      NaN
 75%      NaN
 max      NaN
 Name: rating, dtype: float64,
 Series([], Name: count, dtype: int64))

In [15]:
from pathlib import Path

key = lambda d: d["user"] + "|" + d["item"]
for f in sorted(Path(DATA_DIR).glob("*.csv")):
    if f.name in ("tasks_1_2_train.csv", "task_1_test.csv"):
        continue
    other = load_csv(f.name, DATA_DIR)
    if not {"user", "item"} <= set(other.columns):
        print(f"{f.name:<25} (no user/item columns: {list(other.columns)})")
        continue
    print(f"{f.name:<25} rows {len(other):>8,}  rating col: {'rating' in other.columns!s:<5}  "
          f"T1 test pairs present: {key(test).isin(set(key(other))).mean():6.2%}  "
          f"T1 test users present: {test['user'].isin(other['user']).mean():6.1%}  "
          f"T1 test items present: {test['item'].isin(other['item']).mean():6.1%}")

task_1_predictions.csv    rows   20,000  rating col: True   T1 test pairs present: 100.00%  T1 test users present: 100.0%  T1 test items present: 100.0%
task_2_predictions.csv    (no user/item columns: ['user', 'item1', 'item2', 'which'])
task_2_test.csv           (no user/item columns: ['user', 'item1', 'item2', 'which'])
task_3_predictions.csv    rows   20,000  rating col: True   T1 test pairs present:  0.00%  T1 test users present:   0.0%  T1 test items present:   0.0%
task_3_test.csv           rows   20,000  rating col: True   T1 test pairs present:  0.00%  T1 test users present:   0.0%  T1 test items present:   0.0%
task_4_predictions.csv    (no user/item columns: ['user', 'timestamp', 'item1', 'item2', 'which'])
task_4_test.csv           (no user/item columns: ['user', 'timestamp', 'item1', 'item2', 'which'])
task_5_predictions.csv    (no user/item columns: ['user', 'item1', 'item2', 'which'])
task_5_test.csv           (no user/item columns: ['user', 'item1', 'item2', 'which'])
t